In [ ]:
# H-3 dev training (script 115), Google Colab. Runtime > Change runtime type > T4 GPU.
# Set ARM below, then Runtime > Run all. One arm per session (~5-8 h).
ARM = 'hybrid_mri_rank_mu_s0'   # others: hybrid_mri_rank_enhw_mu_s0, hybrid_mri_rank_mu_s1, hybrid_mri_rank_enhw_mu_s1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/colab_h3'

In [ ]:
import os, shutil, subprocess, sys
for f in ('pg_inputs.npz', 'pg_mri2mm.npz', 'mu_train_2mm.npz', 'test_ids.json'):
    if not os.path.exists('/content/' + f):
        shutil.copy(f'{D}/{f}', '/content/' + f)
shutil.unpack_archive(f'{D}/code.zip', '/content')
!nvidia-smi

In [ ]:
env = dict(os.environ, GBM_PG_PACK='/content/pg_inputs.npz', GBM_PG_TEST_IDS='/content/test_ids.json',
           GBM_MU_PACK='/content/mu_train_2mm.npz', GBM_PG_MRI='/content/pg_mri2mm.npz', PYTHONUNBUFFERED='1', GBM_DISK_CACHE='/content/cache')
p = subprocess.Popen([sys.executable, '-u', 'src/115_predictgbm_unet_dev.py', '--arms', ARM, '--epochs', '60', '--tag', '_h3_' + ARM],
                     cwd='/content/repo', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
print('exit', p.wait())

In [ ]:
import glob
os.makedirs(f'{D}/out', exist_ok=True)
for f in glob.glob('/content/repo/output/predictgbm/*h3*') + glob.glob('/content/repo/output/predictgbm/oof_logits_*.npz'):
    shutil.copy2(f, f'{D}/out/')
print(os.listdir(f'{D}/out'))